# Qwen2.5 (small), from scratch

Goal: implement the architectural pieces that differ between GPT-2 and Qwen2.5, individually, then assemble them into a full small Qwen2 model.

Four differences from `gpt2_small.ipynb` to build one at a time: **RoPE** (rotary positional embeddings, replacing learned `wpe`), **RMSNorm** (replacing `LayerNorm`), **grouped-query attention / GQA** (fewer KV heads than query heads, replacing standard multi-head attention), **SwiGLU** (replacing the GELU-based MLP).

Build order: GQA -> RoPE -> RMSNorm -> SwiGLU (each tested standalone against GPT-2's equivalent piece) -> assemble into the full `Qwen2` model -> (optionally) load real pretrained weights and verify.

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- TODO: GroupedQueryAttention(n_embd, n_head, n_kv_heads) ---
# covering MHA (n_kv_heads == n_head), GQA (1 < n_kv_heads < n_head), MQA (n_kv_heads == 1)


# --- MHA (copied from gpt2_small.ipynb) ---
class CausalSelfAttentionMHA(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.d = n_embd
        self.nh = n_head
        self.dk = self.d// self.nh
        self.c_attn = nn.Linear(self.d, 3 * self.d)
        self.c_proj = nn.Linear(self.d, self.d)
        
    def forward(self, x):
        # x: [B, T, d]
        B, T, _ = x.shape

        ### projection
        x = self.c_attn(x) # [B, T, 3 * d]
        Q, K, V = x.split(self.d, dim=-1) 

        ### reshape
        Q = Q.view(B, T, self.nh, self.dk).transpose(1, 2)
        K = K.view(B, T, self.nh, self.dk).transpose(1, 2)
        V = V.view(B, T, self.nh, self.dk).transpose(1, 2) # [B, n, T, dk]

        ### scaled dot product
        attention_z = Q @ K.transpose(-2, -1)/ (self.dk ** .5) # [B, n, T, T]
        mask = torch.tril(torch.ones(T, T)).bool() 
        attention_mask = attention_z.masked_fill(~mask, float("-inf"))
        attention_scores = torch.softmax(attention_mask, dim=-1)
        out = attention_scores @ V  #   [B, n, T, d] 
        # out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
    
        ### merging
        out = out.transpose(1, 2).contiguous().reshape(B, T, -1) # [B, T, d]
        out = self.c_proj(out) #[B, T, d]

        return out


x = torch.randn([2, 10, 16])
att = CausalSelfAttentionMHA(16, 2)
y = att(x)
print(x.shape, y.shape)


# --- MQA (n_kv_heads == 1) ----
class CausalSelfAttentionMQA(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.d = n_embd
        self.nh = n_head
        self.nhkv = 1
        self.dk = self.d// self.nh
        self.k_proj = nn.Linear(self.d, self.dk)
        self.v_proj = nn.Linear(self.d, self.dk)
        self.q_proj = nn.Linear(self.d, self.d)
        self.c_proj = nn.Linear(self.d, self.d)
        
    def forward(self, x):
        # x: [B, T, d]
        B, T, _ = x.shape

        ### projection
        Q = self.q_proj(x) # [B, T, d]
        K = self.k_proj(x) # [B, T, dk]
        V = self.v_proj(x) # [B, T, dk]
        
        ### reshape
        Q = Q.view(B, T, self.nh, self.dk).transpose(1, 2) # [B, n, T, dk]
        K = K.view(B, T, self.nhkv, self.dk).transpose(1, 2)# [B, 1, T, dk]
        V = V.view(B, T, self.nhkv, self.dk).transpose(1, 2) # [B, 1, T, dk]

        ### scaled dot product
        attention_z = Q @ K.transpose(-2, -1)/ (self.dk ** .5) # [B, n, T, T]
        mask = torch.tril(torch.ones(T, T)).bool() 
        attention_mask = attention_z.masked_fill(~mask, float("-inf"))
        attention_scores = torch.softmax(attention_mask, dim=-1)
        out = attention_scores @ V  #   [B, n, T, d] 
        # out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
    
        ### merging
        out = out.transpose(1, 2).contiguous().reshape(B, T, -1) # [B, T, d]
        out = self.c_proj(out) #[B, T, d]

        return out

x = torch.randn([2, 10, 16])
att = CausalSelfAttentionMQA(16, 2)
y = att(x)
print(x.shape, y.shape)

# --- GQA (n_kv_heads == nhkv) ----
class CausalSelfAttentionGQA(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_heads):
        super().__init__()
        assert n_head % n_kv_heads == 0, "wrong n_kv_heads value!"
        self.d = n_embd
        self.nh = n_head
        self.nhkv = n_kv_heads
        self.dk = self.d// self.nh
        self.k_proj = nn.Linear(self.d, self.nhkv * self.dk)
        self.v_proj = nn.Linear(self.d, self.nhkv * self.dk)
        self.q_proj = nn.Linear(self.d, self.d)
        self.c_proj = nn.Linear(self.d, self.d)
        
    def forward(self, x):
        # x: [B, T, d]
        B, T, _ = x.shape

        ### projection
        Q = self.q_proj(x) # [B, T, d]
        K = self.k_proj(x) # [B, T, dk]
        V = self.v_proj(x) # [B, T, dk]
        
        ### reshape
        Q = Q.view(B, T, self.nh, self.dk).transpose(1, 2) # [B, n, T, dk]
        K = K.view(B, T, self.nhkv, self.dk).transpose(1, 2)# [B, nk, T, dk]
        V = V.view(B, T, self.nhkv, self.dk).transpose(1, 2) # [B, nk, T, dk]

        ### scaled dot product
        K = K.repeat_interleave(self.nh//self.nhkv, dim=1)
        attention_z = Q @ K.transpose(-2, -1)/ (self.dk ** .5) # [B, n, T, T]
        mask = torch.tril(torch.ones(T, T)).bool() 
        attention_mask = attention_z.masked_fill(~mask, float("-inf"))
        attention_scores = torch.softmax(attention_mask, dim=-1)
        V = V.repeat_interleave(self.nh//self.nhkv, dim=1)
        out = attention_scores @ V  #   [B, n, T, d] 
        # out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
    
        ### merging
        out = out.transpose(1, 2).contiguous().reshape(B, T, -1) # [B, T, d]
        out = self.c_proj(out) #[B, T, d]

        return out

x = torch.randn([2, 10, 16])
att = CausalSelfAttentionGQA(16, 4, 2)
y = att(x)
print(x.shape, y.shape)


torch.Size([2, 10, 16]) torch.Size([2, 10, 16])
torch.Size([2, 10, 16]) torch.Size([2, 10, 16])
torch.Size([2, 10, 16]) torch.Size([2, 10, 16])


## RoPE (Rotary Positional Embeddings)

Replaces the learned absolute-position embedding (`wpe`). Absolute position embeddings give the model no structural guarantee that attention scores depend on *relative* distance between tokens — RoPE builds that in directly, by rotating Q and K (not adding a position vector) right before the `Q @ K^T` dot product.

**Row-vector convention** (each row of Q/K is one token's vector, matching real `[T, d]`-shaped tensors):

```
qi_new = qi · R(iθ)ᵀ      (row i of Q, rotated by angle iθ)
kj_new = kj · R(jθ)ᵀ      (row j of K, rotated by angle jθ)
```

**Why the resulting score only depends on relative position** — uses two rotation-matrix facts: rotation matrices are orthogonal, so `R(θ)ᵀ = R(-θ)`; and rotations compose by adding angles, `R(a)R(b) = R(a+b)`.

```
score_ij = qi_new · kj_newᵀ
         = qi · R(iθ)ᵀ · R(jθ) · kjᵀ
         = qi · R(-iθ) · R(jθ) · kjᵀ
         = qi · R((j-i)θ) · kjᵀ
```

Depends only on the relative offset `(j - i)`, never on `i` or `j` individually — that's RoPE's core property, proven directly rather than asserted.

**Implementation shape**: there's no single shared matrix inserted between Q and K (the needed angle `(j-i)θ` differs per `(i,j)` pair — a whole grid, not one fixed matrix). Instead, rotate each row of Q by its own position's angle and each row of K by its own position's angle, independently, *before* the matmul. An ordinary `Q_rot @ K_rot.T` afterward then automatically produces the correct relative-position score at every entry — the relative-position property falls out of the matmul itself, as a byproduct of `R(iθ)ᵀR(jθ) = R((j-i)θ)`, not something explicitly constructed.

Real RoPE doesn't rotate the full `d`-dim vector as one 2D rotation — it splits each head's `dk` dimensions into `dk/2` pairs, and rotates each pair by its own angle (a different base frequency per pair, following the same scheme as sinusoidal position embeddings). The 2D derivation above generalizes pair-by-pair.

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- TODO: add RoPE to this MHA (rotate Q/K rows by position before the score matmul) ---

# --- MHA (copied from gpt2_small.ipynb) ---
class CausalSelfAttentionMHA(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.d = n_embd
        self.nh = n_head
        self.dk = self.d// self.nh
        self.c_attn = nn.Linear(self.d, 3 * self.d)
        self.c_proj = nn.Linear(self.d, self.d)
    

    @staticmethod
    def _rotate(M):
        assert M.dim() == 4, "wrong dimensions"
        B, n, T, dk = M.shape
        new_M = M.clone()
        
        for t in range(T):
            m = M[..., t, :].clone()
            theta = (t * 10000. ** (-2 * torch.arange(dk//2) /dk)).repeat_interleave(2)
            cos_theta = torch.cos(theta).unsqueeze(dim=0).unsqueeze(0)
            sin_theta = torch.sin(theta).unsqueeze(dim=0).unsqueeze(0)
            rotate_half = torch.stack([-m[..., 1::2], m[..., 0::2]], dim=-1).flatten(-2)
            #print(cos_theta.size(), sin_theta.size(), rotate_half.size()) 
            new_M[..., t, :] = m * cos_theta + rotate_half * sin_theta
        return new_M

        
    def forward(self, x):
        # x: [B, T, d]
        B, T, _ = x.shape

        ### projection
        x = self.c_attn(x) # [B, T, 3 * d]
        Q, K, V = x.split(self.d, dim=-1) 

        ### reshape
        Q = Q.view(B, T, self.nh, self.dk).transpose(1, 2)
        Q = self._rotate(Q)
        K = K.view(B, T, self.nh, self.dk).transpose(1, 2)
        K = self._rotate(K)
        V = V.view(B, T, self.nh, self.dk).transpose(1, 2) # [B, n, T, dk]


        ### scaled dot product
        attention_z = Q @ K.transpose(-2, -1)/ (self.dk ** .5) # [B, n, T, T]
        mask = torch.tril(torch.ones(T, T)).bool() 
        attention_mask = attention_z.masked_fill(~mask, float("-inf"))
        attention_scores = torch.softmax(attention_mask, dim=-1)
        out = attention_scores @ V  #   [B, n, T, d] 
        # out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
    
        ### merging
        out = out.transpose(1, 2).contiguous().reshape(B, T, -1) # [B, T, d]
        out = self.c_proj(out) #[B, T, d]

        return out


x = torch.randn([2, 10, 16])
att = CausalSelfAttentionMHA(16, 2)
y = att(x)
print(x.shape, y.shape)

torch.Size([2, 10, 16]) torch.Size([2, 10, 16])


In [ ]:
class SwiGELU(nn.Module):
    def __init__(self, n_embd, hidden):
        super().__init__()
        self.d = n_embd
        self.hidden = hidden
        self.silu = nn.SiLU()
        self.linear_1 = nn.Linear(self.d, self.hidden)
        self.linear_2 = nn.Linear(self.d, self.hidden)
    
    def forward(self, x):
        return self.silu(self.linear_1(x)) * self.linear_2(x)


class MLP(nn.Module):
    def __init__(self, n_embd, intermediate_size):
        super().__init__()
        self.d = n_embd
        self.hidden = intermediate_size
        self.c_proj = nn.Linear(self.hidden, self.d)
        self.swiGELU = SwiGELU(self.d, self.hidden)

    def forward(self, x):
        x = self.swiGELU(x)
        out = self.c_proj(x)
        return out

class Block(nn.Module):
    def __init__(self, n_embd, n_head, intermediate_size):
        super().__init__()
        self.d = n_embd
        self.nh = n_head
        self.hidden = intermediate_size
        self.attn = CausalSelfAttentionMHA(self.d, self.nh)
        self.mlp = MLP(self.d, self.hidden)
        self.ln1 = nn.RMSNorm(self.d)
        self.ln2 = nn.RMSNorm(self.d)

    def forward(self, x):
        x = x + self.attn(self.ln1(x)) 
        x = x + self.mlp(self.ln2(x))
        return x


class Qwen(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocab_size = config.vocab_size
        self.max_len = config.block_size
        self.hidden = config.intermediate_size
        self.d = config.n_embd
        self.nh = config.n_head
        self.nl = config.n_layer
        #self.dk = self.d //self.nh -> come back to this later

        self.wte = nn.Embedding(self.vocab_size, self.d)
        self.ln_f = nn.RMSNorm(self.d)
        self.lm_head = nn.Linear(self.d, self.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        self.h = nn.ModuleList()
        for _ in range(self.nl):
            self.h.append(Block(self.d, self.nh, self.hidden))

    def forward(self, idx):
        B, T = idx.shape
        x = self.wte(idx)
        for i in range(self.nl):
            x = self.h[i](x)
        out = self.lm_head(self.ln_f(x))
        return out
